# SPEI Fraud Detection - Baseline Model

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import xgboost as xgb
from xgboost import XGBClassifier
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.metrics import(
    precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    confusion_matrix, classification_report
)

from src.features.load_data import load_fct_features, load_fraud_typology

In [ ]:
df = load_fct_features()
typology_df = load_fraud_typology()
df = pd.merge(df, typology_df, how='left', on='transaction_id')
df.shape

In [ ]:
df = df.sort_values(by='transaction_date').reset_index(drop=True)
y = df['is_fraud'].astype(int)

cols_to_drop = [
    'transaction_id',
    'sender_account_id',
    'receiver_account_id',
    'transaction_date',
    'is_fraud',
    'fraud_typology'
]

X = df.drop(columns=cols_to_drop)
print(f'Features shape: {X.shape}')
print(f'Target shape: {y.shape}')
print(f'Target Distribution: {y.value_counts(normalize=True).round(4)}')

In [ ]:
sklearn.set_config(transform_output='pandas')

preprocessor = ColumnTransformer(
    transformers=[
        (
            'ordinal_segment', OrdinalEncoder(categories=[['dormant', 'light', 'regular', 'heavy', 'power']]),
            ['sender_activity_segment']
        ),
        (
            'onehot_concept',
            OneHotEncoder(
                max_categories=16,
                handle_unknown='infrequent_if_exist',
                sparse_output=False
            ),
            ['concept_pago']
        ),
    ],
    remainder='passthrough',
    verbose_feature_names_out = False
)

In [ ]:
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', XGBClassifier(scale_pos_weight=67, random_state=42, n_jobs=-1, eval_metric='aucpr')),
])
pipeline

In [ ]:
#80/10/10 split
n = len(df)
train_end = int(n * .8)
val_end = int(n * .9)

X_train, X_val, X_test = X.iloc[:train_end], X.iloc[train_end:val_end], X.iloc[val_end:]
y_train, y_val, y_test = y.iloc[:train_end], y.iloc[train_end:val_end], y.iloc[val_end:]


In [ ]:
test_df = df.iloc[val_end:]
typology_counts = test_df['fraud_typology'].value_counts(dropna=True)
print('Per-typology counts in test:')
print(typology_counts)

In [ ]:
pipeline.fit(X_train, y_train)

In [ ]:
y_val_pred = pipeline.predict(X_val)
y_val_proba = pipeline.predict_proba(X_val)[:, 1]

In [ ]:
cm = confusion_matrix(y_val, y_val_pred)
print("Confusion matrix on val:")
print(pd.DataFrame(cm,
    index=['actual_legit', 'actual_fraud'],
    columns=['pred_legit', 'pred_fraud']))

In [ ]:
precision = precision_score(y_val, y_val_pred)
recall = recall_score(y_val, y_val_pred)
f1 = f1_score(y_val, y_val_pred)
auc_roc = roc_auc_score(y_val, y_val_proba)
auc_pr = average_precision_score(y_val, y_val_proba)

print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1: {f1:.3f}")
print(f"AUC-ROC: {auc_roc:.3f}")
print(f"AUC-PR:{auc_pr:.3f}")

In [ ]:
import joblib
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
MODELS_DIR = PROJECT_ROOT / 'artifacts' / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

model_path = MODELS_DIR / 'baselinexgb.pkl'
joblib.dump(pipeline, model_path)